<a href="https://colab.research.google.com/github/werowe/HypatiaAcademy/blob/master/ml/step_by_step_single_neural_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## A Neural Network with A Single Neuron

This neural network has:
- one neuron
- three input features

The input is:

```python
x = torch.tensor([1.0, 2.0, 3.0])
```

So the neuron receives three values.

Because there are three inputs, we also need:
- three weights

```python
w = torch.rand(3)
```

Each weight is multiplied by one input value:

$
z = w_1x_1 + w_2x_2 + w_3x_3 + b
$

The neuron then:
1. multiplies inputs by weights
2. adds the results together
3. adds the bias term
4. applies the sigmoid activation function

This produces a single prediction: $ŷ$

Even though there are three weights, this is still:
> one neuron

because the entire calculation produces:
- one output
- one prediction
- one probability.

In [2]:
import torch

# input features
x = torch.tensor([1.0, 2.0, 3.0])

# actual observed output value
y = torch.tensor(1.0)

# initialize weights
w = torch.rand(3)

print("\ninitial weights", w)

# initialize bias
b = torch.tensor(0.0)

print("\ninitial bias", b)


initial weights tensor([0.5214, 0.5028, 0.8430])

initial bias tensor(0.)


# Activation Function

The activation function takes the value:

$
z = wx + b
$

and transforms it into an output.

In this example, we use the sigmoid activation function:

$
\sigma(z) = \frac{1}{1 + e^{-z}}
$

The sigmoid function converts any number into a value between 0 and 1.

That is useful because we can interpret the output as a probability.

Examples:

- if $\sigma(z)$ is close to 1, the model predicts “yes”
- if $\sigma(z)$ is close to 0, the model predicts “no”
- if $\sigma(z)=0.5$, the model is uncertain

The sigmoid curve is smooth, not a sharp cutoff.

This smoothness helps the neural network learn gradually during gradient descent.

So the sigmoid function converts the raw score:

$
z = wx + b
$

into a probability prediction $ŷ$

#Sigmoid
Gives a probability between 0 and 1

$σ(z) = \frac{1}{1 + e^{-z}}$

which is the same as :

$σ(z) = \frac{1}{1 + (1/ e^{z})}$

where:

$z = mx + b$

![](https://raw.githubusercontent.com/werowe/HypatiaAcademy/master/images/sigmoid.png)

In [3]:
import torch

def sigmoid(z):
    return 1 / (1 + torch.exp(-z))


# Binary Cross-Entropy Loss


* Rewards Confident Correct Predictions
* Punishes Confident Wrong Predictions

or equivalently:

* Gives low loss to confident correct predictions
* Gives high loss to confident incorrect predictions

The Binary Cross-Entropy formula looks complicated at first:

$
L = -(y\log(\hat{y}) + (1-y)\log(1-\hat{y}))
$

But the basic idea is actually simple.

# The formula has TWO parts

One part handles:
- when the correct answer is 1

The other handles:
- when the correct answer is 0.

---

# Case 1 — Correct answer is 1

Then:

$
y = 1
$

So:

$
1-y = 0
$

That makes the second half disappear.

The formula becomes:

$$
L = -\log(ŷ)
$$

Now the network only cares about:
- how close the prediction is to 1.

---

# Example

If the network predicts:

$
ŷ=0.99
$

then:
- loss is tiny
- very good prediction.

But if:

$
ŷ=0.01
$

then:
- loss becomes huge
- because the model was confidently wrong.

---

# Case 2 — Correct answer is 0

Then:

$
y=0
$

So the FIRST part disappears.

The formula becomes:

$
L=-\log(1-\hat{y})
$

Now the network wants:
- predictions close to 0.

---

# Why the logarithm?

The log function creates:
- gentle penalties for small mistakes
- enormous penalties for confident wrong answers.

That is exactly what we want in classification.

---

# The most important intuition

The formula is basically saying:

> “Reward probabilities close to the correct answer.”
>
> “Punish confident wrong probabilities very strongly.”

That is the core idea of Binary Cross-Entropy Loss.

# Binary Cross-Entropy Loss

The loss function measures:
> how wrong the neural network’s prediction is.

For binary classification:
- the correct answer $y$ is either:
  
$$
0 \quad \text{or} \quad 1
$$

- the prediction $\hat{y}$ is a probability between 0 and 1.

The Binary Cross-Entropy Loss formula is:

$$
L = -(y\log(\hat{y}) + (1-y)\log(1-\hat{y}))
$$

---

# If the Correct Answer is 1

Then:

$$
y = 1
$$

The formula simplifies to:

$$
L = -\log(\hat{y})
$$

This means:
- predictions close to 1 produce small loss
- predictions close to 0 produce large loss

So if the answer is “yes,” the network wants:
- $\hat{y}$ close to 1.

---

# If the Correct Answer is 0

Then:

$$
y = 0
$$

The formula becomes:

$$
L = -\log(1-\hat{y})
$$

This means:
- predictions close to 0 produce small loss
- predictions close to 1 produce large loss

So if the answer is “no,” the network wants:
- $\hat{y}$ close to 0.

---

# Why Use the Logarithm?

The logarithm strongly punishes:
- confident wrong predictions.

Examples:

- predicting:
  
$
ŷ=0.99
$

when the correct answer is 0 produces very large loss.

- predicting:
  
$
ŷ=0.01
$

when the correct answer is 0 produces very small loss.

---

# Intuition

Binary Cross-Entropy Loss rewards:
- confident correct predictions

and heavily penalizes:
- confident incorrect predictions.

That helps the neural network learn better probabilities.

$L(\hat{y}, y) = -\left( y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right)
$

In [4]:
def loss(y_hat, y):
    return -(y * torch.log(y_hat) + (1 - y) * torch.log(1 - y_hat))


In [5]:

def forward_pass(w, x, b, y):
    # calculate z, the linear output
    z = torch.dot(w, x) + b

    # calculate the activation, this is also the prediction y_hat
    y_hat = sigmoid(z)

    # calculate the loss
    l = loss(y_hat, y)

    return y_hat, y, w, b, l


# Adjust Weights and Bias using Gradient

To train the neural network, we need to know how to adjust:
- the weights $w$
- and the bias $b$

to reduce the loss.

The prediction process is:

$
z = wx + b
$

then:

$
ŷ = \sigma(z)
$

The loss tells us how wrong the prediction is.

Backpropagation computes derivatives called gradients that tell us:
> how changing $w$ or $b$ changes the loss.

---

# Bias Gradient

The bias gradient is:

$
\frac{\partial L}{\partial b} = ŷ - y
$

This measures prediction error.

- if $ŷ$ is too large, the gradient is positive
- if $ŷ$ is too small, the gradient is negative

So the bias adjusts in the direction that reduces error.

---

# Weight Gradient

The weight gradient is:

$
\frac{\partial L}{\partial w} = (ŷ - y)x
$

This is similar to the bias gradient, but multiplied by the input value $x$.

Why multiply by $x$?

Because inputs with larger values have a larger effect on the prediction.

So:
- large $x$ → larger weight updates
- small $x$ → smaller weight updates

---

# Intuition

The gradients tell the neural network:

- which direction to move
- and how much to change the parameters

to reduce prediction error.

Gradient descent then updates the parameters repeatedly until the loss becomes small.

In [6]:
def back_propagation(y_hat, y, w, b):
    # gradients
    db = y_hat - y
    dw = (y_hat - y) * x

    # learning rate
    lr = 0.1

    # update weight and bias
    b = b - lr * db
    w = w - lr * dw

    return w, b

# Now Loop until we get Minimum Loss

In [7]:
# how many times to loop
cnt = 100

while cnt > 0:
    y_hat, y, w, b, l = forward_pass(w, x, b, y)

    if cnt % 5 == 0:
        print("loss", l.item())

    w, b = back_propagation(y_hat, y, w, b)

    cnt -= 1


print("\nfinal weights", w)
print("\nfinal bias", b)

print("\nnow make prediction")

y_hat = sigmoid(torch.dot(w, x) + b)

print("\npredicted value y_hat", y_hat.item())
print("\nobserved value y", y.item())

# and since it's logistic regression
if y_hat > 0.5:
    print("\nlogistic regression", True)
else:
    print("\nlogistic regression", False)

loss 0.017169402912259102
loss 0.01521789189428091
loss 0.013664072379469872
loss 0.012397709302604198
loss 0.011345970444381237
loss 0.010458430275321007
loss 0.009699623100459576
loss 0.009043334051966667
loss 0.008470065891742706
loss 0.007965135388076305
loss 0.007516908925026655
loss 0.00711642624810338
loss 0.00675654923543334
loss 0.006431114859879017
loss 0.006135651841759682
loss 0.005866116378456354
loss 0.005619310773909092
loss 0.0053922818042337894
loss 0.005182978697121143
loss 0.004989294335246086

final weights tensor([0.6066, 0.6733, 1.0987])

final bias tensor(0.0852)

now make prediction

predicted value y_hat 0.9952019453048706

observed value y 1.0

logistic regression True
